In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from scipy.stats import norm

In [11]:
X, y = make_classification(n_samples=100000, n_features=4, 
                               n_informative=3, n_redundant=0, 
                               random_state=42)

print(X[:5], y[:5])
    

[[-0.40534235  1.05750977 -0.63609067  0.45745517]
 [ 0.25348987  2.54723793  0.5743398  -0.61182947]
 [-0.97603312  1.72768211 -2.00478339  2.20031652]
 [-2.34804852 -1.18713682 -1.89759384 -0.89952011]
 [ 0.84709555  1.2856912  -0.17194421 -0.47935388]] [1 1 0 0 1]


In [12]:
feature_names = ['Income', 'Debt', 'Age', 'Random_Noise']
df = pd.DataFrame(X, columns=feature_names)
df

,Income,Debt,Age,Random_Noise
0,-0.405342,1.057510,-0.636091,0.457455
1,0.253490,2.547238,0.574340,-0.611829
2,-0.976033,1.727682,-2.004783,2.200317
3,-2.348049,-1.187137,-1.897594,-0.899520
4,0.847096,1.285691,-0.171944,-0.479354
...,...,...,...,...
99995,0.124980,-0.315920,0.129200,0.224967
99996,-0.636623,0.155588,-1.144536,-0.588653
99997,1.167197,1.138608,-0.971237,-1.663077
99998,-0.540449,-0.328067,-0.530091,-0.209727


In [13]:
 # 2. Train XGBoost
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=42)
model = xgb.XGBClassifier(n_estimators=50).fit(X_train, y_train)

# --- FIX: Wrap predict_proba in a lambda to avoid SHAP/XGBoost attribute conflicts ---
model_predict = lambda x: model.predict_proba(x)

# 3. Setup Model-Agnostic Explainer
background = X_train.mean().values.reshape(1, -1)
explainer = shap.KernelExplainer(model_predict, background)

# Pick a specific instance to test
instance = X_test.iloc[0:1]

instance

,Income,Debt,Age,Random_Noise
75721,1.703281,-1.622443,-3.47331,-0.517805


In [14]:
results = []
K =100
n = 95
    
for i, target_feature in enumerate(feature_names):
    # A. Baseline attribution (Class 1)
    shap_values = explainer.shap_values(instance, silent=True)
    # Handle different SHAP output formats (list for multi-class vs array for single)
    e_i = shap_values[0][i] if isinstance(shap_values, list) else shap_values[0, i, 1]
        
    # B. Construct Null Distribution
    null_attributions = []
    # Use a smaller K for the pilot if speed is an issue (e.g., 20 or 30)
    for _ in range(K):
        x_null = instance.copy()
        mu, sigma = df[target_feature].mean(), df[target_feature].std()
        x_null[target_feature] = np.random.normal(mu, sigma)
        
        s_val = explainer.shap_values(x_null, silent=True)
        e_null = s_val[0][i] if isinstance(s_val, list) else s_val[0, i, 1]
        null_attributions.append(abs(e_null))

    print(f"Feature: {target_feature}, Baseline Attribution: {e_i:.4f}, Null Attributions (first 5): {null_attributions[:5]}")
    
    # C. Compute Noise Floor (NF) and Significance Score (S)
    nf_i = np.percentile(null_attributions, n)
    s_i = abs(e_i) / nf_i if nf_i > 0 else 0.0
    admissibility = s_i >= 1.0 
    
    results.append({
        "Feature": target_feature,
        "Attribution": round(e_i, 4),
        "Noise Floor": round(nf_i, 4),
        "S_score": round(s_i, 2),
        "Admissible": admissibility
    })

df_results = pd.DataFrame(results)
print(df_results)

Feature: Income, Baseline Attribution: 0.3012, Null Attributions (first 5): [0.34189434830720233, 0.15417721665774772, 0.294108583436658, 0.4007157754773895, 0.2681023788560803]
Feature: Debt, Baseline Attribution: -0.2711, Null Attributions (first 5): [0.046062745153903885, 0.0012237206101418652, 0.0035627310474714013, 0.006062778333822831, 0.22140926153709495]
Feature: Age, Baseline Attribution: 0.6220, Null Attributions (first 5): [0.041271770489402115, 0.008979041439791492, 0.11074139340780677, 0.011086503431821822, 0.13330397964455187]
Feature: Random_Noise, Baseline Attribution: -0.0017, Null Attributions (first 5): [0.00014599940429138591, 0.003076995878170008, 0.0063994497371215875, 0.0003978461027145386, 0.009805971911797884]
        Feature  Attribution  Noise Floor  S_score  Admissible
0        Income       0.3012       0.5164     0.58       False
1          Debt      -0.2711       0.4022     0.67       False
2           Age       0.6220       0.5131     1.21        True
3  